# Minimal DeepCluster on CIFAR-10

Compare **RGB vs. Sobel** inputs and different cluster counts `K` using ResNet-18.

- CIFAR-10 is downloaded directly with `torchvision.datasets.CIFAR10`.
- DeepCluster alternates between K-means pseudo-label assignment and network training.
- Evaluation: NMI (cluster assignments vs. train labels) and linear-probe Top-1 accuracy.
- W&B is optional. Start with `USE_WANDB = False`.

In [ ]:
# If needed, uncomment:
# %pip install -q torch torchvision scikit-learn wandb

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 
EPOCHS = 10
K_VALUES = [10, 20]
MODES = ["rgb", "sobel"]
NUM_WORKERS = 2
USE_WANDB = True
WANDB_PROJECT = "deepcluster-cifar10"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## CIFAR-10 and input transforms

In [ ]:
!wget -P ./data/  https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz
!tar -xzf ./data/cifar-10-python.tar.gz -C ./dataset/
import tarfile
DATA_DIR = './data'
with tarfile.open('./data/cifar-10-python.tar.gz') as tf:
     tf.extractall(DATA_DIR)
     _found = [DATA_DIR]   # skip download
print("DATA_DIR:", DATA_DIR, "| using local files:", bool(_found))

In [ ]:
class InputTransform:
    def __init__(self, mode="rgb", train=False):
        self.mode = mode
        self.train = train
        self.augment = transforms.Compose([
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
        ]) if train else transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)
        )

    def __call__(self, image):
        x = self.to_tensor(self.augment(image))
        if self.mode == "sobel":
            gray = (0.2989*x[0] + 0.5870*x[1] + 0.1140*x[2])[None, None]
            kx = x.new_tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).view(1, 1, 3, 3)
            ky = x.new_tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).view(1, 1, 3, 3)
            gx = F.conv2d(gray, kx, padding=1)
            gy = F.conv2d(gray, ky, padding=1)
            x = (torch.sqrt(gx.square() + gy.square() + 1e-12) / 4.0).clamp(0, 1)[0]
        return self.normalize(x)


class IndexedCIFAR10(Dataset):
    def __init__(self, base, mode, train):
        self.base = base
        self.transform = InputTransform(mode, train)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, label = self.base.data[idx], self.base.targets[idx]
        from PIL import Image
        image = Image.fromarray(image)
        return self.transform(image), idx, label


train_base = datasets.CIFAR10(root="./data/", train=True, download=False)
test_base = datasets.CIFAR10(root="./data/", train=False, download=False)
y_train = np.asarray(train_base.targets)
y_test = np.asarray(test_base.targets)
NUM_CLASSES = 10

def make_loader(base, mode, train=False, shuffle=False):
    ds = IndexedCIFAR10(base, mode, train)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

print("CIFAR-10:", len(train_base), "train images,", len(test_base), "test images")

In [ ]:
import wandb
if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        wandb.login(key=key) if key else wandb.login()
    except Exception:
        wandb.login()

## Model, feature extraction, and evaluation

In [ ]:
FEATURE_DIM = 512

def build_model(k):
    weights = models.ResNet18_Weights.DEFAULT
    encoder = models.resnet18(weights=None)
    encoder.layer4[0] = nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1, bias=False)
    encoder.fc = nn.Identity()

    class DeepClusterModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = encoder
            self.cluster_head = nn.Linear(FEATURE_DIM, k)

        def forward(self, x):
            return self.cluster_head(self.encoder(x))

    return DeepClusterModel().to(device)


@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    features = np.zeros((len(loader.dataset), FEATURE_DIM), dtype=np.float32)
    labels = np.zeros(len(loader.dataset), dtype=np.int64)
    for x, idx, y in loader:
        z = model.encoder(x.to(device)).cpu().numpy()
        features[idx.numpy()] = z
        labels[idx.numpy()] = y.numpy()
    return features, labels


def cluster_features(features, k):
    x = (features - features.mean(0)) / (features.std(0) + 1e-6)
    return KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(x)


def linear_probe(train_features, test_features):
    scaler = StandardScaler()
    x_train = scaler.fit_transform(train_features)
    x_test = scaler.transform(test_features)
    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    clf.fit(x_train, y_train)
    return clf.score(x_test, y_test)

## Train and compare RGB vs. Sobel

In [ ]:
import wandb
import torch
import torch.nn.functional as F
import pandas as pd
from sklearn.metrics import normalized_mutual_info_score

results = []

for mode in MODES:
    for k in K_VALUES:
        run_name = f"{mode}-k{k}"
        print(f"\n--- mode={mode}, K={k} ---")

        # Initialize W&B once per experiment
        run = None
        if USE_WANDB:
            run = wandb.init(
                project=WANDB_PROJECT,
                name=run_name,
                group=f"deepcluster-{mode}",
                config={
                    "mode": mode,
                    "K": k,
                    "epochs": EPOCHS,
                    "seed": SEED,
                    "optimizer": "SGD",
                    "learning_rate": 0.03,
                    "momentum": 0.9,
                    "weight_decay": 1e-5,
                    "method": "DeepCluster",
                },
                reinit=True,
            )

        try:
            # Deterministic views for clustering/evaluation;
            # augmented views for training.
            train_eval_loader = make_loader( train_base, mode, train=False, shuffle=False )
            train_loader = make_loader( train_base, mode, train=True, shuffle=True)
            test_loader = make_loader(test_base, mode, train=False, shuffle=False )

            model = build_model(k)
            optimizer = torch.optim.SGD( model.parameters(), lr=0.03,momentum=0.9,weight_decay=1e-5, )

            for epoch in range(EPOCHS):
                # Recompute pseudo-labels from current features.
                train_features, labels_seen = extract_features( model, train_eval_loader)
                assignments = cluster_features(train_features, k)
                model.train()
                total_loss, total = 0.0, 0

                for x, idx, _ in train_loader:
                    x = x.to(device)
                    target = torch.as_tensor(assignments[idx.numpy()],dtype=torch.long, device=device)
                    optimizer.zero_grad(set_to_none=True)
                    loss = F.cross_entropy(model(x), target)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item() * len(x)
                    total += len(x)

                epoch_loss = total_loss / total
                print( f"Epoch {epoch + 1:02d}/{EPOCHS} " f"| loss={epoch_loss:.4f}")
                # Log epoch-level metrics
                if run is not None:
                    wandb.log({
                        "epoch": epoch + 1,
                        "train/loss": epoch_loss,
                    }, step=epoch + 1)

            # Final feature extraction
            train_features, train_labels_seen = extract_features(
                model, train_eval_loader
            )
            test_features, test_labels_seen = extract_features(
                model, test_loader
            )

            final_assignments = cluster_features(train_features, k)

            row = {
                "mode": mode,
                "K": k,
                "NMI": normalized_mutual_info_score(
                    train_labels_seen, final_assignments
                ),
                "linear_probe_top1": linear_probe(
                    train_features, test_features
                ),
            }

            results.append(row)
            print(row)

            # Log final metrics to the same run
            if run is not None:
                wandb.log({
                    "final/NMI": row["NMI"],
                    "final/linear_probe_top1": row["linear_probe_top1"],
                    "final/epoch": EPOCHS,
                }, step=EPOCHS + 1)

                wandb.summary["final_NMI"] = row["NMI"]
                wandb.summary["final_linear_probe_top1"] = (
                    row["linear_probe_top1"]
                )

        finally:
            if run is not None:
                run.finish()

# Save results
results_df = pd.DataFrame(results)
display(results_df)

results_df.to_csv("deepcluster_cifar10_results.csv", index=False)
print("Saved deepcluster_cifar10_results.csv")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
model._modules['encoder'].conv1

In [ ]:
filters = model._modules['encoder'].conv1.weight.detach().cpu()

# Arrange 64 filters in an 8x8 grid
fig, axes = plt.subplots(8, 8, figsize=(12, 12))

for i, ax in enumerate(axes.flat):
    # [3, 7, 7] -> [7, 7, 3]
    f = filters[i].permute(1, 2, 0).numpy()[:,:,0]

    # Normalize each filter independently for visualization
    f_min, f_max = f.min(), f.max()
    f = (f - f_min) / (f_max - f_min + 1e-8)

    ax.imshow(f,cmap="gray")
    ax.axis("off")
    ax.set_title(str(i), fontsize=8)

plt.suptitle("ResNet18 conv1: 64 Learned Filters")
plt.tight_layout()
plt.show()
